Imports and definitions

In [4]:
import pandas as pd
import numpy as np
from shapely.geometry import Point, MultiPolygon
import geopandas as gpd
from geopandas import GeoDataFrame
from fuzzywuzzy import process
import geojson_validator



### Merging on Odisha Block Geojson

In [3]:
tenders = pd.read_excel(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\odisha-flood-tenders\Apache file.xlsx', sheet_name=0)
blocks = gpd.read_file(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\odisha-flood-tenders\odisha_block_simplified.geojson')

In [14]:
# Simplify MultiPolygon by selecting the largest Polygon
def simplify_multipolygon(geometry):
    if isinstance(geometry, MultiPolygon):
        # Select the largest Polygon by area
        return max(geometry.geoms, key=lambda geom: geom.area)
    return geometry

# Apply the simplification
blocks['geometry'] = blocks['geometry'].apply(simplify_multipolygon)

In [15]:
fixed_geo = geojson_validator.fix_geometries(blocks)#, optional=["duplicate_nodes"])
#geo_fixed = pd.DataFrame(fixed_geo)


geo_fixed = pd.DataFrame.from_dict(fixed_geo)
geo_fixed

2024-11-15_15:50:35.405 | Criteria 'optional': ['duplicate_nodes']
2024-11-15_15:50:35.406 | Criteria 'invalid': ['unclosed', 'exterior_not_ccw', 'interior_not_cw']
2024-11-15_15:50:35.406 | Criteria 'problematic': ['duplicate_nodes']
2024-11-15_15:50:35.702 | Validation results: {'invalid': {}, 'problematic': {}, 'count_geometry_types': {'Polygon': 314}, 'skipped_validation': []}
2024-11-15_15:50:35.866 | Structure validation results: {'"coordinates" member must be an array, but is a tuple instead': {'path': ['/features/0/geometry', '/features/1/geometry', '/features/2/geometry', '/features/3/geometry', '/features/4/geometry', '/features/5/geometry', '/features/6/geometry', '/features/7/geometry', '/features/8/geometry', '/features/9/geometry', '/features/10/geometry', '/features/11/geometry', '/features/12/geometry', '/features/13/geometry', '/features/14/geometry', '/features/15/geometry', '/features/16/geometry', '/features/17/geometry', '/features/18/geometry', '/features/19/geome

ValueError: All arrays must be of the same length

In [16]:
# Preprocess geometries
blocks['geometry'] = blocks['geometry'].apply(lambda geom: simplify_multipolygon(geom))

# Convert to GeoJSON format
geojson_data = blocks.to_json()

# Validate and fix geometries
fixed_geo = geojson_validator.fix_geometries(geojson_data)


2024-11-15_15:51:12.609 | Criteria 'optional': ['duplicate_nodes']
2024-11-15_15:51:12.610 | Criteria 'invalid': ['unclosed', 'exterior_not_ccw', 'interior_not_cw']
2024-11-15_15:51:12.610 | Criteria 'problematic': ['duplicate_nodes']


ValueError: Filepath or URL must be a geojson or json file

In [11]:
def get_best_match(block_name, block_names):
    match, score = process.extractOne(block_name, block_names)
    return match if score > 80 else None  # Adjust threshold as needed


In [24]:
# List of block names from polygons dataset
block_names = blocks['block_name'].tolist()

# Add a 'Matched_Block' column with best matches
tenders['Matched_Block'] = tenders['BLOCK_FINALISEDD'].apply(lambda x: get_best_match(x, block_names))


In [25]:
merged_gdf = tenders.merge(blocks, left_on='Matched_Block', right_on='block_name', how='left')
merged_gdf = merged_gdf.drop(columns=['dtcode11','block_lgd','dtname','block_name'])
merged_gdf

,tender_title,BLOCK_FINALISEDD,DISTRICT_FINALISEDD,Latitude,Longitude,Pincode,Contract_Dates,Season,Tender_Value,Awarded_Value,...,Product Category,tender_externalreference,Tender ID,Tender Type,Tender Category,EMD Payable To,EMD Payable At,Response Type,Matched_Block,geometry
0,Const. of drain at South side of Siva field in...,ANGUL,ANUGUL,20.84,85.15,759122,2022-05-04,Pre-Monsoon,589286,660000.00,...,Civil Works - Others,EO/AGLM-21/2021-22,2021_ORULB_73437_12,Open Tender,Works,Nil,Nil,Repair and Restoration,ANUGUL,"MULTIPOLYGON (((9438790.24200 2342018.99710, 9..."
1,Construction of bridge cum wire under bridge o...,CHHENDIPADA,ANUGUL,20.84,85.15,759122,2022-06-25,Monsoon,9260164,7872065.00,...,Civil Works - Roads,SE (R and B) Angul-21/ 2021-22,2021_EICCL_74495_1,Open Tender,Works,Nil,Nil,Repair and Restoration,CHHENDIPADA,"MULTIPOLYGON (((9437731.58440 2379589.98440, 9..."
2,Improvement to Pallahara Town road such as wid...,ANGUL,ANUGUL,20.84,85.15,759122,2022-06-25,Monsoon,6091069,5766415.00,...,Civil Works - Roads,EE (R and B) Angul-06/ 2021-22,2021_EICCL_70927_8,Open Tender,Works,Nil,Nil,Others,ANUGUL,"MULTIPOLYGON (((9438790.24200 2342018.99710, 9..."
3,S/R to Angul town road from Aptech Chhaka to R...,ANGUL,ANUGUL,20.84,85.15,759122,2022-06-25,Monsoon,1048346,891199.00,...,Civil Works - Roads,EE (R and B) Angul-10/ 2021-22,2021_EICCL_71932_12,Open Tender,Works,Nil,Nil,Repair and Restoration,ANUGUL,"MULTIPOLYGON (((9438790.24200 2342018.99710, 9..."
4,SR to such as construction of Box cell and Dra...,ANGUL,ANUGUL,20.84,85.15,759122,2022-06-25,Monsoon,1788063,1520032.00,...,Civil Works - Roads,EE (R and B) Angul-12/ 2021-22,2021_EICCL_72246_1,Open Tender,Works,Nil,Nil,Preparedness Measures,ANUGUL,"MULTIPOLYGON (((9438790.24200 2342018.99710, 9..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1155,Construction of Mandabaga Check dam over Local...,SUNDARGARH,SUNDARGARH,22.12,84.04,770014,2023-03-03,Pre-Monsoon,3310831,2814537.32,...,Civil Works - Others,SE MID SNG 05/2022-23,2023_CEMIB_85384_5,Open Tender,Works,Nil,Nil,Repair and Restoration,SUNDARGARH,"POLYGON ((9354040.27590 2505868.32330, 9354421..."
1156,Dismantling and Shifting of Compound wall for ...,SUNDARGARH,SUNDARGARH,22.12,84.04,770001,2023-03-06,Pre-Monsoon,14639052,12444658.00,...,Civil Works - Others,"C.C.E.K.C., Keonjhar No. 28 / 2022-23",2022_EICCL_83413_1,Open Tender,Works,Nil,Nil,Repair and Restoration,SUNDARGARH,"POLYGON ((9354040.27590 2505868.32330, 9354421..."
1157,Improvement to Sambalpur-Sundargarh-Rourkela r...,SUNDARGARH,SUNDARGARH,22.12,84.04,770001,2023-03-06,Pre-Monsoon,32832000,27910552.00,...,Civil Works - Roads,"C.C.E.K.C., Keonjhar No. 26/ 2022-23",2022_EICCL_83064_2,Open Tender,Works,Nil,Nil,Repair and Restoration,SUNDARGARH,"POLYGON ((9354040.27590 2505868.32330, 9354421..."
1158,Construction of Badghumuda Check dam over Brah...,HEMGIR,SUNDARGARH,22.12,84.04,770013,2023-03-28,Pre-Monsoon,5900802,5016271.52,...,Civil Works - Others,SE MID SNG 04/2022-23,2023_CEMIB_85361_3,Open Tender,Works,Nil,Nil,Preparedness Measures,HEMGIR,"POLYGON ((9303955.35650 2491420.38530, 9303995..."


In [3]:
tenders['Block'] = tenders['Block'].str.upper()
tenders.to_csv(r"od_tenders.csv")#['Block'].nunique()

In [3]:
blocks['block_name'] = blocks['block_name'].str.upper()
blocks = blocks.rename(columns={'block_name':'BLOCK'})
blocks

,block_lgd,dtname,dtcode11,BLOCK,geometry
0,3276,Anugul,384,ANUGUL,"MULTIPOLYGON (((9442312.609 2339493.863, 94422..."
1,3277,Anugul,384,ATHMALLIK,"POLYGON ((9390468.652 2364546.332, 9390256.027..."
2,3278,Anugul,384,BANARPAL,"MULTIPOLYGON (((9471339.742 2371642.106, 94715..."
3,3279,Anugul,384,CHHENDIPADA,"MULTIPOLYGON (((9439026.210 2377462.379, 94389..."
4,3280,Anugul,384,KANIHA,"POLYGON ((9460517.606 2393391.650, 9460519.752..."
...,...,...,...,...,...
309,3585,Sundargarh,374,NUAGAON,"POLYGON ((9445371.490 2552607.000, 9445166.112..."
310,3586,Sundargarh,374,RAJGANGPUR,"POLYGON ((9408104.434 2518011.796, 9407754.917..."
311,3587,Sundargarh,374,SUBDEGA,"POLYGON ((9358692.809 2525317.411, 9358660.178..."
312,3588,Sundargarh,374,SUNDARGARH,"POLYGON ((9356755.809 2504146.320, 9356734.676..."


In [4]:
tenders['geometry'] = [Point(xy) for xy in zip(tenders.Longitude, tenders.Latitude)]
tenders_points = GeoDataFrame(tenders, geometry='geometry')
tenders_points = tenders_points.set_crs("EPSG:4326")


In [8]:

# Set CRS for tenders to match blocks
tenders_points = tenders_points.to_crs("EPSG:3857")#set_crs(blocks.crs, allow_override=True)


# Standardize geometry in blocks to MultiPolygon
blocks['geometry'] = blocks['geometry'].apply(
    lambda geom: MultiPolygon([geom]) if geom.type == 'Polygon' else geom
)

# Fix invalid geometries in blocks if any
blocks['geometry'] = blocks['geometry'].buffer(0)

# Perform spatial join using 'intersects' to handle boundary cases
tenderblock = gpd.sjoin(tenders_points, blocks[['BLOCK', 'geometry']], how="left", op="intersects")

tenderblock['geometry'] = tenderblock['index_right'].apply(
    lambda idx: blocks.loc[idx, 'geometry'] if pd.notnull(idx) else None
)

# Drop 'index_right' if no longer needed
tenderblock = tenderblock.drop(columns=['index_right'])

# Set the new geometry as the active geometry column
tenderblock = gpd.GeoDataFrame(tenderblock, geometry='geometry', crs=blocks.crs)

tenderblock


C:\Users\saura\AppData\Local\Temp\ipykernel_30432\1482521914.py:7: ShapelyDeprecationWarning: The 'type' attribute is deprecated, and will be removed in the future. You can use the 'geom_type' attribute instead.
  lambda geom: MultiPolygon([geom]) if geom.type == 'Polygon' else geom
c:\Users\saura\anaconda3\envs\cdl-env\Lib\site-packages\IPython\core\interactiveshell.py:3517: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if await self.run_code(code, result, async_=asy):


,State,Title,District,Block,Pincode,Latitude,Longitude,Organisation_Chain,Bid_Opening_Place,Work_Description,...,Product_Category,Sub_category,Tender_Reference_Number,Tender_ID,Awarded_Value,Contract_Value,Tender_Value,Response_Type,geometry,BLOCK
0,Odisha,Improvement to Pallahara Town road such as wid...,Angul,ANGUL,759122,20.950000,85.266667,EIC-CIVIL||SERNB-CIRCLE-DHENKANAL||EERNB Angul...,O/O the Executive Engineer Angul R and B Division,Improvement to Pallahara Town road such as wid...,...,Civil Works - Roads,NaN,EE (R and B) Angul-06/ 2021-22,2021_EICCL_70927_8,5766415.00,5766415.00,6091069,NaN,"POLYGON ((9493537.528 2378570.161, 9493157.897...",PARJANG
1,Odisha,S/R to Angul town road from Aptech Chhaka to R...,Angul,ANGUL,759122,20.950000,85.266667,EIC-CIVIL||SERNB-CIRCLE-DHENKANAL||EERNB Angul...,O/O the Executive Engineer Angul R and B Division,S/R to Angul town road from Aptech Chhaka to R...,...,Civil Works - Roads,NaN,EE (R and B) Angul-10/ 2021-22,2021_EICCL_71932_12,891199.00,891199.00,1048346,NaN,"POLYGON ((9493537.528 2378570.161, 9493157.897...",PARJANG
2,Odisha,S/R to NH 157 B to Helei via Belataila road su...,Angul,ANGUL,759122,20.950000,85.266667,EIC-CIVIL||SERNB-CIRCLE-DHENKANAL||EERNB Angul...,O/O the Executive Engineer Angul R and B Division,S/R to NH 157 B to Helei via Belataila road su...,...,Civil Works - Roads,NaN,EE (R and B) Angul-10/ 2021-22,2021_EICCL_71932_7,751225.00,751225.00,883690,NaN,"POLYGON ((9493537.528 2378570.161, 9493157.897...",PARJANG
3,Odisha,Construction of Boxcell Culvert at 32.900 km o...,Angul,ANGUL,759122,20.950000,85.266667,EIC-CIVIL||SERNB-CIRCLE-DHENKANAL||EERNB Angul...,O/O the Executive Engineer Angul R and B Division,Construction of Boxcell Culvert at 32.900 km o...,...,Civil Works - Roads,NaN,EE (R and B) Angul-12/ 2021-22,2021_EICCL_72245_3,6377510.00,6377510.00,7502070,NaN,"POLYGON ((9493537.528 2378570.161, 9493157.897...",PARJANG
4,Odisha,SR to such as construction of Box cell and Dra...,Angul,ANGUL,759122,20.950000,85.266667,EIC-CIVIL||SERNB-CIRCLE-DHENKANAL||EERNB Angul...,O/O the Executive Engineer Angul R and B Division,SR to such as construction of Box cell and Dra...,...,Civil Works - Roads,NaN,EE (R and B) Angul-12/ 2021-22,2021_EICCL_72246_1,1520032.00,1520032.00,1788063,NaN,"POLYGON ((9493537.528 2378570.161, 9493157.897...",PARJANG
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1150,Odisha,Construction of Sikipani-II Check dam over Kad...,Sundargarh,NaN,770001,22.416667,85.000000,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Sikipani-II Check dam over Kad...,...,Civil Works - Others,Check dam,SE MID SNG 05/2022-23,2023_CEMIB_85384_3,3110610.28,3110610.28,3659111,NaN,"POLYGON ((9445371.490 2552607.000, 9445166.112...",NUAGAON
1151,Odisha,Construction of Gaijornalla Check dam over Gai...,Sundargarh,NaN,770001,22.416667,85.000000,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Gaijornalla Check dam over Gai...,...,Civil Works - Others,Check dam,SE MID SNG 05/2022-23,2023_CEMIB_85384_4,2981385.59,2981385.59,3507100,NaN,"POLYGON ((9445371.490 2552607.000, 9445166.112...",NUAGAON
1152,Odisha,Construction of Mandabaga Check dam over Local...,Sundargarh,NaN,770014,22.416667,85.000000,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Mandabaga Check dam over Local...,...,Civil Works - Others,Check dam,SE MID SNG 05/2022-23,2023_CEMIB_85384_5,2814537.32,2814537.32,3310831,NaN,"POLYGON ((9445371.490 2552607.000, 9445166.112...",NUAGAON
1153,Odisha,Construction of Tihuria-III Check dam over Tum...,Sundargarh,NaN,770013,22.416667,85.000000,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Tihuria-III Check dam over Tum...,...,Civil Works - Others,Check dam,SE MID SNG 05/2022-

In [9]:
tenderblock.to_csv(r'od_tender_blocks.csv')


### replacing polygon geometry with strings

In [1]:
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon
import numpy as np
import pandas as pd
import geojson_validator


In [26]:

def geometry_to_text(geometry):
    if geometry is None or geometry.is_empty:
        return None  # Return None for empty or missing geometries
    elif isinstance(geometry, Polygon):
        # Convert Polygon to a list of vertices
        return ', '.join([f"({x}, {y})" for x, y in geometry.exterior.coords])
    elif isinstance(geometry, MultiPolygon):
        # Convert MultiPolygon to vertices, handling each polygon in the MultiPolygon
        return ' | '.join(
            [', '.join([f"({x}, {y})" for x, y in polygon.exterior.coords]) for polygon in geometry.geoms]
        )
    else:
        return None  # Handle any other geometry types if necessary


#fixed_geo = fixed_geo.explode(index_parts=True)
#fixed_geo = []
# Apply the function to create a new column
merged_gdf['polygons'] = merged_gdf['geometry'].apply(geometry_to_text)

merged_gdf.drop('geometry', axis=1).to_csv(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\odisha-flood-tenders\od_tenders_apache_polygon.csv')

## Merging on Odisha pincodes

In [2]:
tenders = pd.read_excel(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\odisha-flood-tenders\Odisha flood tender 10th oct 2024.xlsx', sheet_name=0)
pincodes = pd.read_excel(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\odisha-flood-tenders\Odisha flood tender 10th oct 2024.xlsx', sheet_name=2)

In [3]:
pincodes['PINCODE'].nunique()

550

##### pincodes has 551 unique pincodes

In [4]:
tenders

,State,Title,District,Block,Pincode,Latitude,Longitude,Organisation_Chain,Bid_Opening_Place,Work_Description,Season,Contract_Date,Product_Category,Sub_category,Tender_Reference_Number,Tender_ID,Awarded_Value,Contract_Value,Tender_Value,Response_Type
0,Odisha,Improvement to Pallahara Town road such as wid...,Angul,Angul,759122,20.950000,85.266667,EIC-CIVIL||SERNB-CIRCLE-DHENKANAL||EERNB Angul...,O/O the Executive Engineer Angul R and B Division,Improvement to Pallahara Town road such as wid...,Monsoon,2022-06-25,Civil Works - Roads,NaN,EE (R and B) Angul-06/ 2021-22,2021_EICCL_70927_8,5766415.00,5766415.00,6091069,NaN
1,Odisha,S/R to Angul town road from Aptech Chhaka to R...,Angul,Angul,759122,20.950000,85.266667,EIC-CIVIL||SERNB-CIRCLE-DHENKANAL||EERNB Angul...,O/O the Executive Engineer Angul R and B Division,S/R to Angul town road from Aptech Chhaka to R...,Monsoon,2022-06-25,Civil Works - Roads,NaN,EE (R and B) Angul-10/ 2021-22,2021_EICCL_71932_12,891199.00,891199.00,1048346,NaN
2,Odisha,S/R to NH 157 B to Helei via Belataila road su...,Angul,Angul,759122,20.950000,85.266667,EIC-CIVIL||SERNB-CIRCLE-DHENKANAL||EERNB Angul...,O/O the Executive Engineer Angul R and B Division,S/R to NH 157 B to Helei via Belataila road su...,Monsoon,2022-06-26,Civil Works - Roads,NaN,EE (R and B) Angul-10/ 2021-22,2021_EICCL_71932_7,751225.00,751225.00,883690,NaN
3,Odisha,Construction of Boxcell Culvert at 32.900 km o...,Angul,Angul,759122,20.950000,85.266667,EIC-CIVIL||SERNB-CIRCLE-DHENKANAL||EERNB Angul...,O/O the Executive Engineer Angul R and B Division,Construction of Boxcell Culvert at 32.900 km o...,Monsoon,2022-06-26,Civil Works - Roads,NaN,EE (R and B) Angul-12/ 2021-22,2021_EICCL_72245_3,6377510.00,6377510.00,7502070,NaN
4,Odisha,SR to such as construction of Box cell and Dra...,Angul,Angul,759122,20.950000,85.266667,EIC-CIVIL||SERNB-CIRCLE-DHENKANAL||EERNB Angul...,O/O the Executive Engineer Angul R and B Division,SR to such as construction of Box cell and Dra...,Monsoon,2022-06-25,Civil Works - Roads,NaN,EE (R and B) Angul-12/ 2021-22,2021_EICCL_72246_1,1520032.00,1520032.00,1788063,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1150,Odisha,Construction of Sikipani-II Check dam over Kad...,Sundargarh,NaN,770001,22.416667,85.000000,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Sikipani-II Check dam over Kad...,Post-Monsoon,2023-02-27,Civil Works - Others,Check dam,SE MID SNG 05/2022-23,2023_CEMIB_85384_3,3110610.28,3110610.28,3659111,NaN
1151,Odisha,Construction of Gaijornalla Check dam over Gai...,Sundargarh,NaN,770001,22.416667,85.000000,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Gaijornalla Check dam over Gai...,Post-Monsoon,2023-02-27,Civil Works - Others,Check dam,SE MID SNG 05/2022-23,2023_CEMIB_85384_4,2981385.59,2981385.59,3507100,NaN
1152,Odisha,Construction of Mandabaga Check dam over Local...,Sundargarh,NaN,770014,22.416667,85.000000,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Mandabaga Check dam over Local...,Pre-Monsoon,2023-03-03,Civil Works - Others,Check dam,SE MID SNG 05/2022-23,2023_CEMIB_85384_5,2814537.32,2814537.32,3310831,NaN
1153,Odisha,Construction of Tihuria-III Check dam over Tum...,Sundargarh,NaN,770013,22.416667,85.000000,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Tihuria-III Check dam over Tum...,Post-Monsoon,2023-02-27,Civil Works - Others,Check dam,SE MID SNG 05/2022-23,2023_CEMIB_85384_6,3627615.07,3627615.07,4267280,NaN


In [42]:

# Merge both datasets on the 'pincode' column, keeping all rows from the main dataframe (left join)
merged_df = pd.merge(tenders, pincodes[['PINCODE', 'BLOCK']], on='Pincode', how='left', suffixes=('', '_ref'))

# Fill missing 'block' values in df_main with the corresponding values from df_ref
merged_df['Block'] = merged_df['Block'].fillna(merged_df['BLOCK'])

# Drop the extra 'block_ref' column
merged_df = merged_df.drop(columns=['block_ref'])

# Identify entries where there was no match for the pincode (i.e., block is still missing)
no_match = merged_df[merged_df['block'].isna()]

# Save the updated dataset to a new Excel file
merged_df#.to_excel('flood_tenders_updated.xlsx', index=False)

# Save the no-match cases for further investigation
#no_match.to_excel('no_match_pincodes.xlsx', index=False)

KeyError: 'Pincode'

In [5]:
pincodes = pincodes.dropna(subset='PINCODE')

In [7]:
blocks_na = tenders[tenders['Block'].isna()]#loc[tenders['Pincode'].isna()]
blocks_na

,State,Title,District,Block,Pincode,Latitude,Longitude,Organisation_Chain,Bid_Opening_Place,Work_Description,Season,Contract_Date,Product_Category,Sub_category,Tender_Reference_Number,Tender_ID,Awarded_Value,Contract_Value,Tender_Value,Response_Type
89,Odisha,RETROFITTING OF RPWS TO CHAMPU WITH 50 KL RCC ...,Balasore,NaN,756001,21.500000,86.9,Rural Water Supply and Sanitation||RWSS Circle...,O/O EE RWSS DIVISION BALASORE,RPWS WITH RCC ESR,Post-Monsoon,2022-11-07,Civil Works - Water Works,NaN,TCN NO. 21 DT. 16-08-2021,2021_RWSS_70514_1,5065952.00,5065952.00,5628210,NaN
90,Odisha,RETROFITTING OF RPWS TO RANASAHI UNDER BALASOR...,Balasore,NaN,756001,21.500000,86.9,Rural Water Supply and Sanitation||RWSS Circle...,O/O EE RWSS DIVISION BALASORE,RPWS WORKS,Post-Monsoon,2022-11-07,Civil Works - Water Works,NaN,TCN NO. 25 DT. 22.10.2021,2021_RWSS_72210_1,5812182.00,5812182.00,6837056,NaN
91,Odisha,Periodical Maintenance of Thanachhak Deula PWD...,Balasore,NaN,756086,21.500000,86.9,CE RW I||RWCIRCLE-BALASORE||RWDIVISION-JALESWAR,R.W.Division Jaleswar,Periodical Maintenance of Thanachhak Deula PWD...,Monsoon,2022-07-28,Civil Works - Roads,NaN,TCN No.01/22-23,2022_CERWI_79128_1,758807.00,892610.00,892610,NaN
92,Odisha,Repair and Restoration of Gunsartha Kiagadia P...,Balasore,NaN,756086,21.500000,86.9,CE RW I||RWCIRCLE-BALASORE||RWDIVISION-JALESWAR,Rural Works Division Jaleswar,Road Work,Post-Monsoon,2023-02-15,Civil Works - Roads,NaN,SE/RW/Jls- 04/2022-23,2022_CERWI_83508_26,446445.00,525168.00,525618,NaN
93,Odisha,Repair and Restoration of Batagram Chitrarekha...,Balasore,NaN,756086,21.500000,86.9,CE RW I||RWCIRCLE-BALASORE||RWDIVISION-JALESWAR,Rural Works Division Jaleswar,Road Work,Post-Monsoon,2023-02-16,Civil Works - Roads,NaN,SE/RW/Jls- 04/2022-23,2022_CERWI_83508_30,443594.00,521814.00,521814,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1150,Odisha,Construction of Sikipani-II Check dam over Kad...,Sundargarh,NaN,770001,22.416667,85.0,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Sikipani-II Check dam over Kad...,Post-Monsoon,2023-02-27,Civil Works - Others,Check dam,SE MID SNG 05/2022-23,2023_CEMIB_85384_3,3110610.28,3110610.28,3659111,NaN
1151,Odisha,Construction of Gaijornalla Check dam over Gai...,Sundargarh,NaN,770001,22.416667,85.0,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Gaijornalla Check dam over Gai...,Post-Monsoon,2023-02-27,Civil Works - Others,Check dam,SE MID SNG 05/2022-23,2023_CEMIB_85384_4,2981385.59,2981385.59,3507100,NaN
1152,Odisha,Construction of Mandabaga Check dam over Local...,Sundargarh,NaN,770014,22.416667,85.0,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Mandabaga Check dam over Local...,Pre-Monsoon,2023-03-03,Civil Works - Others,Check dam,SE MID SNG 05/2022-23,2023_CEMIB_85384_5,2814537.32,2814537.32,3310831,NaN
1153,Odisha,Construction of Tihuria-III Check dam over Tum...,Sundargarh,NaN,770013,22.416667,85.0,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Tihuria-III Check dam over Tum...,Post-Monsoon,2023-02-27,Civil Works - Others,Check dam,SE MID SNG 05/2022-23,2023_CEMIB_85384_6,3627615.07,3627615.07,4267280,NaN


In [11]:
blocks_na['Pincode'] = blocks_na['Pincode'].astype(str)
pincodes['PINCODE'] = pincodes['PINCODE'].astype(str)

pincodes = pincodes.drop_duplicates(subset='PINCODE')

# Merge the two DataFrames on the "Pincode" column, keeping all rows from the tenders dataframe
merged_df = pd.merge(blocks_na, pincodes[['PINCODE', 'BLOCK']], 
                     left_on='Pincode', right_on='PINCODE', how='left')

# Fill missing 'Block' values in df_tenders with values from the reference 'BLOCK' column
merged_df['Block'] = merged_df['Block'].fillna(merged_df['BLOCK'])

# Drop the extra 'BLOCK' and 'PINCODE' columns after merging
merged_df = merged_df.drop(columns=['BLOCK', 'PINCODE'])
merged_df['Block'] = merged_df['Block'].fillna('Unknown')
#merged_df = merged_df = merged_df.dropna(subset='Block')
merged_df

C:\Users\saura\AppData\Local\Temp\ipykernel_31464\2423656788.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  blocks_na['Pincode'] = blocks_na['Pincode'].astype(str)


,State,Title,District,Block,Pincode,Latitude,Longitude,Organisation_Chain,Bid_Opening_Place,Work_Description,Season,Contract_Date,Product_Category,Sub_category,Tender_Reference_Number,Tender_ID,Awarded_Value,Contract_Value,Tender_Value,Response_Type
0,Odisha,RETROFITTING OF RPWS TO CHAMPU WITH 50 KL RCC ...,Balasore,BALASORE,756001,21.500000,86.9,Rural Water Supply and Sanitation||RWSS Circle...,O/O EE RWSS DIVISION BALASORE,RPWS WITH RCC ESR,Post-Monsoon,2022-11-07,Civil Works - Water Works,NaN,TCN NO. 21 DT. 16-08-2021,2021_RWSS_70514_1,5065952.00,5065952.00,5628210,NaN
1,Odisha,RETROFITTING OF RPWS TO RANASAHI UNDER BALASOR...,Balasore,BALASORE,756001,21.500000,86.9,Rural Water Supply and Sanitation||RWSS Circle...,O/O EE RWSS DIVISION BALASORE,RPWS WORKS,Post-Monsoon,2022-11-07,Civil Works - Water Works,NaN,TCN NO. 25 DT. 22.10.2021,2021_RWSS_72210_1,5812182.00,5812182.00,6837056,NaN
2,Odisha,Periodical Maintenance of Thanachhak Deula PWD...,Balasore,Unknown,756086,21.500000,86.9,CE RW I||RWCIRCLE-BALASORE||RWDIVISION-JALESWAR,R.W.Division Jaleswar,Periodical Maintenance of Thanachhak Deula PWD...,Monsoon,2022-07-28,Civil Works - Roads,NaN,TCN No.01/22-23,2022_CERWI_79128_1,758807.00,892610.00,892610,NaN
3,Odisha,Repair and Restoration of Gunsartha Kiagadia P...,Balasore,Unknown,756086,21.500000,86.9,CE RW I||RWCIRCLE-BALASORE||RWDIVISION-JALESWAR,Rural Works Division Jaleswar,Road Work,Post-Monsoon,2023-02-15,Civil Works - Roads,NaN,SE/RW/Jls- 04/2022-23,2022_CERWI_83508_26,446445.00,525168.00,525618,NaN
4,Odisha,Repair and Restoration of Batagram Chitrarekha...,Balasore,Unknown,756086,21.500000,86.9,CE RW I||RWCIRCLE-BALASORE||RWDIVISION-JALESWAR,Rural Works Division Jaleswar,Road Work,Post-Monsoon,2023-02-16,Civil Works - Roads,NaN,SE/RW/Jls- 04/2022-23,2022_CERWI_83508_30,443594.00,521814.00,521814,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1061,Odisha,Construction of Sikipani-II Check dam over Kad...,Sundargarh,Sundargarh,770001,22.416667,85.0,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Sikipani-II Check dam over Kad...,Post-Monsoon,2023-02-27,Civil Works - Others,Check dam,SE MID SNG 05/2022-23,2023_CEMIB_85384_3,3110610.28,3110610.28,3659111,NaN
1062,Odisha,Construction of Gaijornalla Check dam over Gai...,Sundargarh,Sundargarh,770001,22.416667,85.0,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Gaijornalla Check dam over Gai...,Post-Monsoon,2023-02-27,Civil Works - Others,Check dam,SE MID SNG 05/2022-23,2023_CEMIB_85384_4,2981385.59,2981385.59,3507100,NaN
1063,Odisha,Construction of Mandabaga Check dam over Local...,Sundargarh,Balisanakara,770014,22.416667,85.0,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Mandabaga Check dam over Local...,Pre-Monsoon,2023-03-03,Civil Works - Others,Check dam,SE MID SNG 05/2022-23,2023_CEMIB_85384_5,2814537.32,2814537.32,3310831,NaN
1064,Odisha,Construction of Tihuria-III Check dam over Tum...,Sundargarh,Hemgir,770013,22.416667,85.0,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Tihuria-III Check dam over Tum...,Post-Monsoon,2023-02-27,Civil Works - Others,Check dam,SE MID SNG 05/2022-23,2023_CEMIB_85384_6,3627615.07,3627615.07,4267280,NaN


In [12]:
merged_df['Block'] = merged_df['Block'].str.title()
tagged = tenders.dropna(subset=['Block'])
filled = pd.concat([tagged,merged_df])
filled

,State,Title,District,Block,Pincode,Latitude,Longitude,Organisation_Chain,Bid_Opening_Place,Work_Description,Season,Contract_Date,Product_Category,Sub_category,Tender_Reference_Number,Tender_ID,Awarded_Value,Contract_Value,Tender_Value,Response_Type
0,Odisha,Improvement to Pallahara Town road such as wid...,Angul,Angul,759122,20.950000,85.266667,EIC-CIVIL||SERNB-CIRCLE-DHENKANAL||EERNB Angul...,O/O the Executive Engineer Angul R and B Division,Improvement to Pallahara Town road such as wid...,Monsoon,2022-06-25,Civil Works - Roads,NaN,EE (R and B) Angul-06/ 2021-22,2021_EICCL_70927_8,5766415.00,5766415.00,6091069,NaN
1,Odisha,S/R to Angul town road from Aptech Chhaka to R...,Angul,Angul,759122,20.950000,85.266667,EIC-CIVIL||SERNB-CIRCLE-DHENKANAL||EERNB Angul...,O/O the Executive Engineer Angul R and B Division,S/R to Angul town road from Aptech Chhaka to R...,Monsoon,2022-06-25,Civil Works - Roads,NaN,EE (R and B) Angul-10/ 2021-22,2021_EICCL_71932_12,891199.00,891199.00,1048346,NaN
2,Odisha,S/R to NH 157 B to Helei via Belataila road su...,Angul,Angul,759122,20.950000,85.266667,EIC-CIVIL||SERNB-CIRCLE-DHENKANAL||EERNB Angul...,O/O the Executive Engineer Angul R and B Division,S/R to NH 157 B to Helei via Belataila road su...,Monsoon,2022-06-26,Civil Works - Roads,NaN,EE (R and B) Angul-10/ 2021-22,2021_EICCL_71932_7,751225.00,751225.00,883690,NaN
3,Odisha,Construction of Boxcell Culvert at 32.900 km o...,Angul,Angul,759122,20.950000,85.266667,EIC-CIVIL||SERNB-CIRCLE-DHENKANAL||EERNB Angul...,O/O the Executive Engineer Angul R and B Division,Construction of Boxcell Culvert at 32.900 km o...,Monsoon,2022-06-26,Civil Works - Roads,NaN,EE (R and B) Angul-12/ 2021-22,2021_EICCL_72245_3,6377510.00,6377510.00,7502070,NaN
4,Odisha,SR to such as construction of Box cell and Dra...,Angul,Angul,759122,20.950000,85.266667,EIC-CIVIL||SERNB-CIRCLE-DHENKANAL||EERNB Angul...,O/O the Executive Engineer Angul R and B Division,SR to such as construction of Box cell and Dra...,Monsoon,2022-06-25,Civil Works - Roads,NaN,EE (R and B) Angul-12/ 2021-22,2021_EICCL_72246_1,1520032.00,1520032.00,1788063,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1061,Odisha,Construction of Sikipani-II Check dam over Kad...,Sundargarh,Sundargarh,770001,22.416667,85.000000,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Sikipani-II Check dam over Kad...,Post-Monsoon,2023-02-27,Civil Works - Others,Check dam,SE MID SNG 05/2022-23,2023_CEMIB_85384_3,3110610.28,3110610.28,3659111,NaN
1062,Odisha,Construction of Gaijornalla Check dam over Gai...,Sundargarh,Sundargarh,770001,22.416667,85.000000,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Gaijornalla Check dam over Gai...,Post-Monsoon,2023-02-27,Civil Works - Others,Check dam,SE MID SNG 05/2022-23,2023_CEMIB_85384_4,2981385.59,2981385.59,3507100,NaN
1063,Odisha,Construction of Mandabaga Check dam over Local...,Sundargarh,Balisanakara,770014,22.416667,85.000000,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Mandabaga Check dam over Local...,Pre-Monsoon,2023-03-03,Civil Works - Others,Check dam,SE MID SNG 05/2022-23,2023_CEMIB_85384_5,2814537.32,2814537.32,3310831,NaN
1064,Odisha,Construction of Tihuria-III Check dam over Tum...,Sundargarh,Hemgir,770013,22.416667,85.000000,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Tihuria-III Check dam over Tum...,Post-Monsoon,2023-02-27,Civil Works - Others,Check dam,SE MID SNG 05/2022-23,2023_CEMIB_85384_6,3627615.07,3627615.07,4267280,NaN


In [29]:
filled['Tender_Reference_Number'].dtype

dtype('O')

In [53]:
#merged = blocks_na.merge(pincodes, left_on='Pincode', right_on='PINCODE', how='left')
merged = pd.merge(
    left=blocks_na, 
    right=pincodes,
    how='left',
    left_on=['Pincode'],# 'Block'],
    right_on=['PINCODE'], #'BLOCK'],
)
merged = merged.dropna(subset='PINCODE')
merged

,State,Title,District,Block,Pincode,Latitude,Longitude,Organisation_Chain,Bid_Opening_Place,Work_Description,...,OFFICE NAME,OFFICE STATUS,PINCODE,TELEPHONE NO.,BLOCK,DISTRICT,STATE/U.T.,POSTAL DIVISION,POSTAL REGION,POSTAL CIRCLE
0,Odisha,RETROFITTING OF RPWS TO CHAMPU WITH 50 KL RCC ...,Balasore,NaN,756001,21.500000,86.9,Rural Water Supply and Sanitation||RWSS Circle...,O/O EE RWSS DIVISION BALASORE,RPWS WITH RCC ESR,...,Azimabad S.O,Sub Post Office,756001,06782-262701,BALASORE,BALESWAR,ORISSA,BALASORE,BHUBANESWAR HQ,ORISSA
1,Odisha,RETROFITTING OF RPWS TO CHAMPU WITH 50 KL RCC ...,Balasore,NaN,756001,21.500000,86.9,Rural Water Supply and Sanitation||RWSS Circle...,O/O EE RWSS DIVISION BALASORE,RPWS WITH RCC ESR,...,Balasore Court S.O,Sub Post Office,756001,06782-262067,BALASORE,BALESWAR,ORISSA,BALASORE,BHUBANESWAR HQ,ORISSA
2,Odisha,RETROFITTING OF RPWS TO CHAMPU WITH 50 KL RCC ...,Balasore,NaN,756001,21.500000,86.9,Rural Water Supply and Sanitation||RWSS Circle...,O/O EE RWSS DIVISION BALASORE,RPWS WITH RCC ESR,...,Balasore H.O,Head Post Office,756001,06782-262310,Balasore,BALESWAR,ORISSA,BALASORE,BHUBANESWAR HQ,ORISSA
3,Odisha,RETROFITTING OF RPWS TO CHAMPU WITH 50 KL RCC ...,Balasore,NaN,756001,21.500000,86.9,Rural Water Supply and Sanitation||RWSS Circle...,O/O EE RWSS DIVISION BALASORE,RPWS WITH RCC ESR,...,Balasore RS S.O,Sub Post Office,756001,06782-262159,BALASORE,BALESWAR,ORISSA,BALASORE,BHUBANESWAR HQ,ORISSA
4,Odisha,RETROFITTING OF RPWS TO CHAMPU WITH 50 KL RCC ...,Balasore,NaN,756001,21.500000,86.9,Rural Water Supply and Sanitation||RWSS Circle...,O/O EE RWSS DIVISION BALASORE,RPWS WITH RCC ESR,...,Fakirmohan College S.O,Sub Post Office,756001,06782-262290,BALASORE,BALESWAR,ORISSA,BALASORE,BHUBANESWAR HQ,ORISSA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5932,Odisha,Construction of Kund Check dam over Chhatenjor...,Sundargarh,NaN,770013,22.416667,85.0,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Kund Check dam over Chhatenjor...,...,Garjanjore B.O,Branch Post Office,770013,NaN,Hemgir,SUNDERGARH,ORISSA,SUNDARGARH,SAMBALPUR,ORISSA
5933,Odisha,Construction of Kund Check dam over Chhatenjor...,Sundargarh,NaN,770013,22.416667,85.0,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Kund Check dam over Chhatenjor...,...,Hemgir S.O,Sub Post Office,770013,NaN,Hemgir,SUNDERGARH,ORISSA,SUNDARGARH,SAMBALPUR,ORISSA
5934,Odisha,Construction of Kund Check dam over Chhatenjor...,Sundargarh,NaN,770013,22.416667,85.0,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Kund Check dam over Chhatenjor...,...,Kendudihi B.O,Branch Post Office,770013,NaN,Hemgir,SUNDERGARH,ORISSA,SUNDARGARH,SAMBALPUR,ORISSA
5935,Odisha,Construction of Kund Check dam over Chhatenjor...,Sundargarh,NaN,770013,22.416667,85.0,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Kund Check dam over Chhatenjor...,...,Sanghumunda B.O,Branch Post Office,770013,NaN,Hemgir,SUNDERGARH,ORISSA,SUNDARGARH,SAMBALPUR,ORISSA


### Tagging response type

In [13]:
import pandas as pd
import os
import re
import dateutil.parser
import glob

In [24]:

# Identify flood related tenders using keywords
def populate_keyword_dict(keyword_list): 
        keywords_dict = {}
        for keyword in keyword_list:
            keywords_dict[keyword] = 0
        return keywords_dict

def flood_filter(row):
    '''
    :param row: row of the dataframe that contains tender title, work description
    
    :return: Tuple of (is_flood_tender, positive_kw_dict, negative_kw_dict) for every row
    '''
    positive_keywords_dict = populate_keyword_dict(POSITIVE_KEYWORDS)
    negative_keywords_dict = populate_keyword_dict(NEGATIVE_KEYWORDS)
    tender_slug = str(row['Tender_Reference_Number']) + ' ' + str(row['Title']) + ' ' + str(row['Work_Description'])
    tender_slug = re.sub(r'[^a-zA-Z0-9 \n\.]', ' ', tender_slug)
    
    is_flood_tender = False
    for keyword in POSITIVE_KEYWORDS:
        keyword_count = len(re.findall(r"\b%s\b" % keyword.lower(), tender_slug.lower()))
        positive_keywords_dict[keyword] = keyword_count
        if keyword_count > 0:
            is_flood_tender = True
            
    for keyword in NEGATIVE_KEYWORDS:
        keyword_count = len(re.findall(r"\b%s\b" % keyword.lower(), tender_slug.lower()))
        negative_keywords_dict[keyword] = keyword_count
        if keyword_count > 0:
            is_flood_tender = False
           
    return str(is_flood_tender), str(positive_keywords_dict), str(negative_keywords_dict)



In [33]:
input_df = filled

input_df['Tender_Reference_Number'] = input_df['Tender_Reference_Number'].astype(str)
input_df['Title'] = input_df['Title'].astype(str)
input_df['Work_Description'] = input_df['Work_Description'].astype(str)

#Flood Keywords
global POSITIVE_KEYWORDS
POSITIVE_KEYWORDS = ['Flood', 'Embankment', 'embkt', 'Relief', 'Erosion', 'SDRF', 'Inundation', 'Hydrology',
                'Silt', 'Siltation', 'Bund', 'Trench', 'Breach', 'Culvert', 'Sluice', 'Dyke',
                'Storm water drain','Emergency','Immediate', 'IM', 'AE','A E', 'AAPDA MITRA']
global NEGATIVE_KEYWORDS
NEGATIVE_KEYWORDS = ['Floodlight', 'Flood Light','GAS', 'FIFA', 'pipe','pipes', 'covid']

flood_filter_tuples = input_df.apply(flood_filter,axis=1)
input_df.loc[:,'is_flood_tender'] = [var[0] for var in list(flood_filter_tuples)]
input_df.loc[:,'positive_keywords_dict'] = [var[1] for var in list(flood_filter_tuples)]
input_df.loc[:,'negative_keywords_dict'] = [var[2] for var in list(flood_filter_tuples)]

# Removing tenders from certain departments that are not related to flood management.
idea_frm_tenders_df = input_df#[(input_df.is_flood_tender=='True')&
                               # (~input_df.Department.isin(["Directorate of Agriculture and Assam Seed Corporation","Department of Handloom Textile and Sericulture"]))]

print('Number of flood related tenders filtered: ', idea_frm_tenders_df.shape[0])
#if idea_frm_tenders_df.shape[0]==0:
#    continue

# Classify tenders based on Monsoons
for index, row in idea_frm_tenders_df.iterrows():
    monsoon = "" 
    published_date = dateutil.parser.parse(row['Contract_Date'])
    if 1 <= published_date.month <= 5:
        monsoon = "Pre-Monsoon"
        if published_date.month == 5 and published_date.day > 14:
            monsoon = "Monsoon"
    elif 6 <= published_date.month <= 10:
        monsoon = "Monsoon"
        if published_date.month == 10 and published_date.day > 14:
            monsoon = "Post-Monsoon"
    else:
        monsoon = "Post-Monsoon"
    idea_frm_tenders_df.loc[index, "Season"] = monsoon

# identify scheme related information
schemes_identified = []
scheme_kw = {'ridf','sdrf','sopd','cidf','ltif'}
for idx, row in idea_frm_tenders_df.iterrows():
    tender_slug = row['Title']+' '+row['Tender_Reference_Number']+' '+row['Work_Description']
    tender_slug = re.sub(r'[^a-zA-Z0-9 \n\.]', ' ', tender_slug).lower()

    tender_slug = set(re.split(r'[-.,()_\s/]\s*',tender_slug))
    try:
        schemes_identified.append(list(tender_slug & scheme_kw)[0].upper())
    except:
        schemes_identified.append('')

idea_frm_tenders_df.loc[:,'Scheme'] = schemes_identified

# EROSION RELATED TENDERS
EROSION_KEYWORDS = ['anti erosion', 'ae', 'a/e', 'a e', 'erosion', 'eroded', 'erroded', 'errosion']
for index, row in idea_frm_tenders_df.iterrows():
    tender_slug = str(row['Tender_Reference_Number']) + ' ' + str(row['Title']) + ' ' + str(row['Work_Description'])
    tender_slug = re.sub(r'[^a-zA-Z0-9 \n\.]', ' ', tender_slug)

    is_present = [len(re.findall(r"\b%s\b" % kw.lower(), tender_slug.lower())) for kw in EROSION_KEYWORDS]
    if sum(is_present)>0:
        idea_frm_tenders_df.loc[index, "Erosion"] = True
    else:
        idea_frm_tenders_df.loc[index, "Erosion"] = False

# ROADS, BRIDGES EMBANKMENTS RELATED TENDERS
ROADS_BRIDGES_EMBANKMENTS_KEYWORDS = ['roads', 'bridges', 'road', 'bridge', 'storm water drain' ,'drain',
                                        'box cul', 'box culvert', 'box culv', 'culvert' ,'embankment', 'embkt',
                                        'river bank protection', 'bund', 'bunds', 'bundh', 'bank protection', 'dyke',
                                        'dyke wall', 'dyke walls', 'silt', 'siltation', 'sluice', 'breach']
for index, row in idea_frm_tenders_df.iterrows():
    tender_slug = str(row['Tender_Reference_Number']) + ' ' + str(row['Title']) + ' ' + str(row['Work_Description'])
    tender_slug = re.sub(r'[^a-zA-Z0-9 \n\.]', ' ', tender_slug)

    is_present = [len(re.findall(r"\b%s\b" % kw.lower(), tender_slug.lower())) for kw in ROADS_BRIDGES_EMBANKMENTS_KEYWORDS]
    if sum(is_present)>0:
        idea_frm_tenders_df.loc[index, "Roads_Bridges_Embkt"] = True
    else:
        idea_frm_tenders_df.loc[index, "Roads_Bridges_Embkt"] = False

#Classification of Tenders based on Response Type
IMMEDIATE_MEASURES_KEYWORDS = ['sdrf','im','i/m','gr','g/r','relief','package','pkt','immediate', 'emergency', 'pk', 'g.r.', 'i.m.']
REPAIR_RESTORATION_IMPROVEMENTS_KEYWORDS = ['improvement', 'imp.', 'impvt', 'impt.', 'repair',
                                            'repairing', 'restoration', 'reconstruction', 'reconstn', 'recoupment',
                                            'raising', 'strengthening', 'r/s', 'm and r', 'upgradation', 'renovation',
                                            'repairing/renovation', 'up-gradation', 'm-r', 'm-r ', 'mr', 'widening', 'r s', 'extension',
                                            'replacement', 're-shaping', 're-grading']
# PREPAREDNESS_MEASURES_KEYWORDS = ['protection','new', 'reconstruction', 'constn' ,'recoupment', 'restoration', 'embankment', 'embkt',
#                     'dyke','culvert','storm water', 'drainage','drain','drains','box','rcc','silt','desiltation','prosiltation',
#                     'anti erosion', 'erosion','a/e','ae','a e','bank protection','bank breach','breach','sludging','desludging',
#                     'sluice','bund','bundh', 'dam','canal','road','roads',
#                     'bridge','bridges','data','drone','rescue','consultation','advisory','consult','study']

PREPAREDNESS_KEYWORDS = ['shelter', 'shelters', 'tarpaulin', 'shelter ',
                            'responder kit', 'aapda mitra volunteers','aapda mitra volunteer', 'district emergency stockpile', 'search light',
                            'life buoys', 'boat ambulances', 'boat ambulance', 'inflatable rubber',
                            'mechanized boats', 'mechanised boats','mechanized boat', 'mechanised boat']
for index, row in idea_frm_tenders_df.iterrows():
    immedidate_measures_dict = populate_keyword_dict(IMMEDIATE_MEASURES_KEYWORDS)
    repair_restoration_dict = populate_keyword_dict(REPAIR_RESTORATION_IMPROVEMENTS_KEYWORDS)
    preparedness_measures_dict = populate_keyword_dict(PREPAREDNESS_KEYWORDS)
    
    response_type = "Others"
    tender_slug = str(row['Tender_Reference_Number']) + ' ' + str(row['Title']) + ' ' + str(row['Work_Description'])
    tender_slug = re.sub(r'[^a-zA-Z0-9 \n\.]', ' ', tender_slug)
    
    for keyword in immedidate_measures_dict:
        keyword_count = len(re.findall(r"\b%s\b" % keyword.lower(), tender_slug.lower()))
        immedidate_measures_dict[keyword] = keyword_count
        if not keyword_count:
            immedidate_measures_dict[keyword] =  False
        else:
            response_type = "Immediate Measures"

    for keyword in repair_restoration_dict:
        keyword_count = len(re.findall(r"\b%s\b" % keyword.lower(), tender_slug.lower()))
        repair_restoration_dict[keyword] = keyword_count
        if not keyword_count:
            repair_restoration_dict[keyword] =  False
        else:
            response_type = "Repair and Restoration"
    
    for keyword in preparedness_measures_dict:
        keyword_count = len(re.findall(r"\b%s\b" % keyword.lower(), tender_slug.lower()))
        preparedness_measures_dict[keyword] = keyword_count
        if not keyword_count:
            preparedness_measures_dict[keyword] =  False
        elif response_type == "Others":
            response_type = "Preparedness Measures"
    idea_frm_tenders_df.loc[index, "Response Type"] = response_type
    
    if response_type == "Immediate Measures":
        sub_head_dict = {k: v for k, v in immedidate_measures_dict.items() if v is not False}
        idea_frm_tenders_df.loc[index, "Flood Response - Subhead"] = str(sub_head_dict)
    elif response_type == "Repair and Restoration":
        sub_head_dict = {k: v for k, v in repair_restoration_dict.items() if v is not False}
        idea_frm_tenders_df.loc[index, "Flood Response - Subhead"] = str(sub_head_dict) 
    elif response_type == "Preparedness Measures":
        sub_head_dict = {k: v for k, v in preparedness_measures_dict.items() if v is not False}
        idea_frm_tenders_df.loc[index, "Flood Response - Subhead"] = str(sub_head_dict)  


    
    idea_frm_tenders_df.to_csv(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\odisha-flood-tenders\testoutput\tagged.csv',
                            encoding='utf-8',
                            index=False)

Number of flood related tenders filtered:  1155


KeyError: 'Contract_Date'

In [31]:
idea_frm_tenders_df

,State,Title,District,Block,Pincode,Latitude,Longitude,Organisation_Chain,Bid_Opening_Place,Work_Description,...,Tender_Value,Response_Type,is_flood_tender,positive_keywords_dict,negative_keywords_dict,Scheme,Erosion,Roads_Bridges_Embkt,Response Type,Flood Response - Subhead
0,Odisha,Improvement to Pallahara Town road such as wid...,Angul,Angul,759122,20.950000,85.266667,EIC-CIVIL||SERNB-CIRCLE-DHENKANAL||EERNB Angul...,O/O the Executive Engineer Angul R and B Division,Improvement to Pallahara Town road such as wid...,...,6091069,NaN,True,"{'Flood': 0, 'Embankment': 0, 'embkt': 0, 'Rel...","{'Floodlight': 0, 'Flood Light': 0, 'GAS': 0, ...",,False,False,Others,"{'improvement': 4, 'widening': 2}"
1,Odisha,S/R to Angul town road from Aptech Chhaka to R...,Angul,Angul,759122,20.950000,85.266667,EIC-CIVIL||SERNB-CIRCLE-DHENKANAL||EERNB Angul...,O/O the Executive Engineer Angul R and B Division,S/R to Angul town road from Aptech Chhaka to R...,...,1048346,NaN,False,"{'Flood': 0, 'Embankment': 0, 'embkt': 0, 'Rel...","{'Floodlight': 0, 'Flood Light': 0, 'GAS': 0, ...",,False,False,Others,NaN
2,Odisha,S/R to NH 157 B to Helei via Belataila road su...,Angul,Angul,759122,20.950000,85.266667,EIC-CIVIL||SERNB-CIRCLE-DHENKANAL||EERNB Angul...,O/O the Executive Engineer Angul R and B Division,S/R to NH 157 B to Helei via Belataila road su...,...,883690,NaN,False,"{'Flood': 0, 'Embankment': 0, 'embkt': 0, 'Rel...","{'Floodlight': 0, 'Flood Light': 0, 'GAS': 0, ...",,False,True,Others,NaN
3,Odisha,Construction of Boxcell Culvert at 32.900 km o...,Angul,Angul,759122,20.950000,85.266667,EIC-CIVIL||SERNB-CIRCLE-DHENKANAL||EERNB Angul...,O/O the Executive Engineer Angul R and B Division,Construction of Boxcell Culvert at 32.900 km o...,...,7502070,NaN,True,"{'Flood': 0, 'Embankment': 0, 'embkt': 0, 'Rel...","{'Floodlight': 0, 'Flood Light': 0, 'GAS': 0, ...",,False,True,Repair and Restoration,"{'repair': 1, 'restoration': 1}"
4,Odisha,SR to such as construction of Box cell and Dra...,Angul,Angul,759122,20.950000,85.266667,EIC-CIVIL||SERNB-CIRCLE-DHENKANAL||EERNB Angul...,O/O the Executive Engineer Angul R and B Division,SR to such as construction of Box cell and Dra...,...,1788063,NaN,False,"{'Flood': 0, 'Embankment': 0, 'embkt': 0, 'Rel...","{'Floodlight': 0, 'Flood Light': 0, 'GAS': 0, ...",,False,True,Repair and Restoration,"{'repair': 1, 'restoration': 1}"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1061,Odisha,Construction of Sikipani-II Check dam over Kad...,Sundargarh,Sundargarh,770001,22.416667,85.000000,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Sikipani-II Check dam over Kad...,...,3659111,NaN,False,"{'Flood': 0, 'Embankment': 0, 'embkt': 0, 'Rel...","{'Floodlight': 0, 'Flood Light': 0, 'GAS': 0, ...",,False,False,Others,NaN
1062,Odisha,Construction of Gaijornalla Check dam over Gai...,Sundargarh,Sundargarh,770001,22.416667,85.000000,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Gaijornalla Check dam over Gai...,...,3507100,NaN,False,"{'Flood': 0, 'Embankment': 0, 'embkt': 0, 'Rel...","{'Floodlight': 0, 'Flood Light': 0, 'GAS': 0, ...",,False,False,Others,NaN
1063,Odisha,Construction of Mandabaga Check dam over Local...,Sundargarh,Balisanakara,770014,22.416667,85.000000,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Mandabaga Check dam over Local...,...,3310831,NaN,False,"{'Flood': 0, 'Embankment': 0, 'embkt': 0, 'Rel...","{'Floodlight': 0, 'Flood Light': 0, 'GAS': 0, ...",,False,False,Others,NaN
1064,Odisha,Construction of Tihuria-III Check dam over Tum...,Sundargarh,Hemgir,770013,22.416667,85.000000,CEMinor IrrigationBBSR||SENMIC-SAMBALPUR||SUND...,O/o the Superintending Engineer MID Sundargarh,Construction of Tihuria-III Check dam over Tum...,...,4267280,NaN,False,"{'Flood': 0, 'Embankment'

In [ ]:
idea_frm_tenders_df

### India Post pincode list

In [3]:
tagged =pd.read_csv(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\odisha-flood-tenders\testoutput\tagged.csv')
indiapost = pd.read_csv(r"D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\flood-data-ecosystem-Odisha\Maps\all_india_PO_list_without_APS_offices_ver2.csv")
odisha = indiapost.loc[indiapost['statename']=='ODISHA']
odisha

,officename,pincode,officeType,Deliverystatus,divisionname,regionname,circlename,Taluk,Districtname,statename,Telephone,Related Suboffice,Related Headoffice
91044,A.K.Deuli B.O,761111,B.O,Delivery,Aska,Berhampur,Odisha,Aska,Ganjam,ODISHA,NaN,Nuagam S.O,Aska H.O
91045,Adapada B.O,761144,B.O,Delivery,Aska,Berhampur,Odisha,Digapahandi,Ganjam,ODISHA,NaN,Konkarada S.O,Aska H.O
91046,Adipur B.O,761118,B.O,Delivery,Aska,Berhampur,Odisha,Buguda,Ganjam,ODISHA,NaN,Buguda S.O,Bhanjanagar H.O
91047,Aladi B.O,761120,B.O,Delivery,Aska,Berhampur,Odisha,Bhanjanagar,Ganjam,ODISHA,NaN,Baragam S.O,Bhanjanagar H.O
91048,Alasuguma B.O,761121,B.O,Delivery,Aska,Berhampur,Odisha,Bhanjanagar,Ganjam,ODISHA,NaN,Jagannath Prasad S.O,Bhanjanagar H.O
...,...,...,...,...,...,...,...,...,...,...,...,...,...
99205,Udusu B.O,770036,B.O,Delivery,Sundargarh,Sambalpur,Odisha,Bisra,Sundergarh,ODISHA,NaN,Bisra S.O,Rourkela H.O
99206,Ujalpur S.O,770011,S.O,Delivery,Sundargarh,Sambalpur,Odisha,Lephripara,Sundergarh,ODISHA,NaN,NaN,Sundargarh H.O
99207,Ushra Colony B.O,770034,B.O,Delivery,Sundargarh,Sambalpur,Odisha,Rajagangapur,Sundergarh,ODISHA,NaN,Kansbahal S.O,Uditnagar H.O
99208,Vedvyas B.O,769004,B.O,Delivery,Sundargarh,Sambalpur,Odisha,Raghunathapali,Sundergarh,ODISHA,NaN,Rourkela - 4 S.O,Uditnagar H.O


In [4]:
import pandas as pd
from fuzzywuzzy import fuzz

In [8]:
df = pd.merge(odisha, tagged, left_on='pincode',right_on='Pincode')

# Function to find the closest match in col2 for each entry in col1
def find_best_match(entry, comparison_column):
    best_match = None
    best_score = 0
    for comparison_entry in comparison_column:
        score = fuzz.ratio(entry, comparison_entry)
        if score > best_score:
            best_score = score
            best_match = comparison_entry
    return best_match, best_score

# Apply function to col1
#df['best_match'], df['similarity_score'] = zip(*tagged['Block'].apply(lambda x: find_best_match(x, indiapost['Taluk'])))

# View results
print(df)
df.to_csv(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\odisha-flood-tenders\testoutput\tagged_test.csv')

          officename  pincode officeType Deliverystatus divisionname  \
0      A.K.Deuli B.O   761111        B.O       Delivery         Aska   
1      A.K.Deuli B.O   761111        B.O       Delivery         Aska   
2      A.K.Deuli B.O   761111        B.O       Delivery         Aska   
3      A.K.Deuli B.O   761111        B.O       Delivery         Aska   
4      A.K.Deuli B.O   761111        B.O       Delivery         Aska   
...              ...      ...        ...            ...          ...   
9385  Sundargarh H.O   770001        H.O       Delivery   Sundargarh   
9386  Sundargarh H.O   770001        H.O       Delivery   Sundargarh   
9387  Sundargarh H.O   770001        H.O       Delivery   Sundargarh   
9388      Talita B.O   770038        B.O       Delivery   Sundargarh   
9389  Tangargaon B.O   770014        B.O       Delivery   Sundargarh   

     regionname circlename           Taluk Districtname statename  ...  \
0     Berhampur     Odisha            Aska       Ganjam    OD

In [10]:
df['Taluk'].nunique()

195

In [7]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Create TF-IDF vectors
vectorizer = TfidfVectorizer()
tfidf_matrix1 = vectorizer.fit_transform(df['Block'])
tfidf_matrix2 = vectorizer.transform(df['Taluk'])

# Calculate cosine similarity
cosine_similarities = cosine_similarity(tfidf_matrix1, tfidf_matrix2)

# Add cosine similarity to the dataframe
df['cosine_similarity'] = cosine_similarities.diagonal()

print(df)

          officename  pincode officeType Deliverystatus divisionname  \
0      A.K.Deuli B.O   761111        B.O       Delivery         Aska   
1      A.K.Deuli B.O   761111        B.O       Delivery         Aska   
2      A.K.Deuli B.O   761111        B.O       Delivery         Aska   
3      A.K.Deuli B.O   761111        B.O       Delivery         Aska   
4      A.K.Deuli B.O   761111        B.O       Delivery         Aska   
...              ...      ...        ...            ...          ...   
9385  Sundargarh H.O   770001        H.O       Delivery   Sundargarh   
9386  Sundargarh H.O   770001        H.O       Delivery   Sundargarh   
9387  Sundargarh H.O   770001        H.O       Delivery   Sundargarh   
9388      Talita B.O   770038        B.O       Delivery   Sundargarh   
9389  Tangargaon B.O   770014        B.O       Delivery   Sundargarh   

     regionname circlename           Taluk Districtname statename  ...  \
0     Berhampur     Odisha            Aska       Ganjam    OD

### Tagging response Type

In [1]:
import pandas as pd
import os
import re
import dateutil.parser
import glob

In [2]:
tender_block = pd.read_csv(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\odisha-flood-tenders\floodtenders_blockgeotagged.csv')
tender_block

,Unnamed: 0,tender_title,location,Pincode,Organisation Chain,Bid Opening Place,Work Description,Tender Value in ₹,Product Category,Sub category,...,tender_district_externalReference,tender_district_title_description,tender_district_location,DISTRICT_FINALISED,tender_villages,tender_block,tender_subdistrict,gp,tender_block_location,BLOCK_FINALISED
0,146,Construction of bridge cum wire under bridge o...,ANGUL,759122,EIC-CIVIL||SERNB-CIRCLE-DHENKANAL||EERNB Angul...,O/O the Superintending Engineer Angul RB Division,Construction of bridge cum wire under bridge o...,"92,60,164",Civil Works - Roads,NaN,...,NaN,Anugul,NaN,Anugul,"'KOSALA', 'CHHENDIPADA'",CHHENDIPADA,NaN,KOSALA,CHHENDIPADA,CHHENDIPADA
1,168,Construction of c.c. drain from Mahendra Basti...,Angul,759122,Municipal Bodies||Angul Municipality,"O/O Executive Officer, Angul Municipality, Angul",Construction of c.c. drain from Mahendra Basti...,"2,73,290",Civil Works - Others,NaN,...,NaN,Anugul,NaN,Anugul,NaN,NaN,NaN,NaN,CHHENDIPADA,CHHENDIPADA
2,292,Construction of Matellia IV Check Dam in Banar...,"Banarpal, Angul",759128,"CE,Minor Irrigation,BBSR||SESMIC-BERHAMPUR",ACE CMIC Bhubaneswar,Construction of Matellia IV Check Dam in Banar...,"1,25,36,000",Civil Works - Others,NaN,...,NaN,NaN,Anugul,Anugul,NaN,NaN,NaN,NaN,CHHENDIPADA,CHHENDIPADA
3,711,Improvement to service road of LBC from RD 0.0...,"Samal, Angul, Odisha",759037,CE and BM Brahmani Basin Samal||APD cum CCE RI...,"Office of the ACE, RHWC, Samal",Improvement to service road of LBC from RD 0.0...,"4,54,54,208",Civil Works - Roads,NaN,...,NaN,Anugul,NaN,Anugul,'SAMAL',PALALAHADA,NaN,NaN,CHHENDIPADA,CHHENDIPADA
4,1018,Retrofitting of 70LPCD for RPWS to village Bon...,Angul,759100,Rural Water Supply and Sanitation||RWSS Circle...,"E.E, RWSS Division, Angul at Talcher",Retrofitting of 70LPCD for RPWS to village Bon...,"3,67,62,000",Civil Works - Water Works,NaN,...,NaN,Anugul,NaN,Anugul,"'RAJIBPUR', 'JHAJIRIBAHAL', 'TULASIPAL', 'CHAU...",BANARPAL,NaN,TULASIPAL,BANARPAL,BANARPAL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1099,726,Installation of Recharge Shafts and Monitoring...,Deputy Director Geology GWD Division Cuttack,753003,Ground Water Survey and Investigation,CUTTACK,Installation of Recharge Shafts and Monitoring...,"42,13,000",Civil Works - Others,NaN,...,NaN,Kendrapara,Cuttack,CONFLICT,NaN,NaN,NaN,NaN,NaN,NaN
1100,727,Installation of Recharge Shafts and Monitoring...,"GWD Division ,Cuttack , New Zobra",753003,Ground Water Survey and Investigation,CUTTACK,Installation of Recharge Shafts and Monitoring...,"42,13,000",Civil Works - Others,NaN,...,NaN,Kendrapara,Cuttack,CONFLICT,NaN,NaN,NaN,NaN,NaN,NaN
1101,783,Protection to left bank river Bahuda near vill...,Berhampur_K Nuagaon,760004,"CE-BM,RVN Basin,Berhampur||S.E. Southern Irr. ...","O/o the SE, Chikiti Irrigation Division, Berha...","Earth Work, Concrete Work, stone work","5,19,821",Civil Works - Others,NaN,...,NaN,Ganjam,Baleshwar,CONFLICT,NaN,NaN,NaN,NaN,NaN,NaN
1102,803,Protection to river bank of river Bahuda near ...,Berhampur_Bajragumma,760004,"CE-BM,RVN Basin,Berhampur||S.E. Southern Irr. ...","O/o the SE, Chikiti Irrigation Division, Berha...","Earth Work, Concrete Work, Stone work","35,40,068",Civil Works - Others,NaN,...,NaN,Ganjam,Baleshwar,CONFLICT,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
def populate_keyword_dict(keyword_list): 
    keywords_dict = {}
    for keyword in keyword_list:
        keywords_dict[keyword] = 0
    return keywords_dict

#Classification of Tenders based on Response Type
IMMEDIATE_MEASURES_KEYWORDS = ['sdrf','im','i/m','gr','g/r','relief','package','pkt','immediate', 'emergency', 'pk', 'g.r.', 'i.m.']

REPAIR_RESTORATION_IMPROVEMENTS_KEYWORDS = ['Check Dam','Construction','Bridge','Retrofitting',
                                        'Drain',
                                        'improvement', 'imp.', 'impvt', 'impt.', 'repair',
                                        'repairing', 'restoration', 'reconstruction', 'reconstn', 'recoupment',
                                        'raising', 'strengthening', 'r/s', 'm and r', 'upgradation', 'renovation',
                                        'repairing/renovation', 'up-gradation', 'm-r', 'm-r ', 'mr', 'widening', 'r s', 'extension',
                                        'replacement', 're-shaping', 're-grading',
                                        'check dam', 'construction', 'bridge', 'retrofitting', 'drain']


PREPAREDNESS_KEYWORDS = ['per','Period','Periodical maintainance','Maintainance','Annual maintenance','Protection','scoured','Scoured bank','Recharge Shaft','De-weeding','Cleaning ','Flood Protection work',
                        'shelter', 'shelters', 'tarpaulin', 'shelter ',
                        'responder kit', 'aapda mitra volunteers','aapda mitra volunteer', 'district emergency stockpile', 'search light',
                        'life buoys', 'boat ambulances', 'boat ambulance', 'inflatable rubber',
                        'mechanized boats', 'mechanised boats','mechanized boat', 'mechanised boat'
                        'rain shelter', 'periodical maintainance','water proofing', 'maintenance', 'annual maintenance', 'protection', 'scoured', 'recharge', 'cleaning ','Flood Protection work',]
for index, row in tender_block.iterrows():
    immedidate_measures_dict = populate_keyword_dict(IMMEDIATE_MEASURES_KEYWORDS)
    repair_restoration_dict = populate_keyword_dict(REPAIR_RESTORATION_IMPROVEMENTS_KEYWORDS)
    preparedness_measures_dict = populate_keyword_dict(PREPAREDNESS_KEYWORDS)

    response_type = "Others"
    tender_slug = str(row['tender_externalreference']) + ' ' + str(row['tender_title']) + ' ' + str(row['Work Description'])
    tender_slug = re.sub(r'[^a-zA-Z0-9 \n\.]', ' ', tender_slug)
    
    for keyword in preparedness_measures_dict:
        keyword_count = len(re.findall(r"\b%s\b" % keyword.lower(), tender_slug.lower()))
        preparedness_measures_dict[keyword] = keyword_count
        if not keyword_count:
            preparedness_measures_dict[keyword] =  False
        elif response_type == "Others":
            response_type = "Preparedness Measures"

    for keyword in immedidate_measures_dict:
        keyword_count = len(re.findall(r"\b%s\b" % keyword.lower(), tender_slug.lower()))
        immedidate_measures_dict[keyword] = keyword_count
        if not keyword_count:
            immedidate_measures_dict[keyword] =  False
        else:
            response_type = "Immediate Measures"

    for keyword in repair_restoration_dict:
        keyword_count = len(re.findall(r"\b%s\b" % keyword.lower(), tender_slug.lower()))
        repair_restoration_dict[keyword] = keyword_count
        if not keyword_count:
            repair_restoration_dict[keyword] =  False
        else:
            response_type = "Repair and Restoration"

    
    tender_block.loc[index, "Response Type"] = response_type

    if response_type == "Immediate Measures":
        sub_head_dict = {k: v for k, v in immedidate_measures_dict.items() if v is not False}
        tender_block.loc[index, "Flood Response - Subhead"] = str(sub_head_dict)
    elif response_type == "Repair and Restoration":
        sub_head_dict = {k: v for k, v in repair_restoration_dict.items() if v is not False}
        tender_block.loc[index, "Flood Response - Subhead"] = str(sub_head_dict) 
    elif response_type == "Preparedness Measures":
        sub_head_dict = {k: v for k, v in preparedness_measures_dict.items() if v is not False}
        tender_block.loc[index, "Flood Response - Subhead"] = str(sub_head_dict)  

tender_block.to_csv(r'tender_block_scheme.csv')

In [ ]:

# input_df - after the scraper code is run
data_path = os.getcwd() + r'/flood-data-ecosystem-Himachal-Pradesh/Sources/TENDERS/data/monthly_tenders/'

csvs = glob.glob(data_path+'*.csv')

for csv in csvs:
    filename  = re.split(r'/',csv)[-1]
    filename  = re.split(r'\\',csv)[-1]
    print ("FILENAME"+ filename)
    input_df = pd.read_csv(csv)
    
    # De-Duplication (Change the logic once the time of scraping is added in the input_df)
    input_df = input_df.drop_duplicates()
    tender_ids = input_df["Tender ID"]
    # duplicates_df = input_df[tender_ids.isin(tender_ids[tender_ids.duplicated()])].sort_values("Tender ID")
    # input_df = input_df.drop(duplicates_df[duplicates_df['No of Bids Received'].isnull()].index)
    # input_df.reset_index(drop=True, inplace=True)
    # deduped_df = input_df.drop_duplicates(subset=['Tender ID'],keep='last')
    # deduped_df.to_csv(os.getcwd()+'/Sources/TENDERS/data/deduped_master_tender_list.csv', encoding='utf-8')

    #Flood Keywords
    global POSITIVE_KEYWORDS
    POSITIVE_KEYWORDS = ['Flood', 'Embankment', 'embkt', 'Relief', 'Erosion', 'SDRF', 'Inundation', 'Hydrology',
                    'Silt', 'Siltation', 'Bund', 'Trench', 'Breach', 'Culvert', 'Sluice', 'Dyke',
                    'Storm water drain','Emergency','Immediate', 'IM', 'AE','A E', 'AAPDA MITRA']
    global NEGATIVE_KEYWORDS
    NEGATIVE_KEYWORDS = ['Floodlight', 'Flood Light','GAS', 'FIFA', 'pipe','pipes', 'covid']

    flood_filter_tuples = input_df.apply(flood_filter,axis=1)
    input_df.loc[:,'is_flood_tender'] = [var[0] for var in list(flood_filter_tuples)]
    input_df.loc[:,'positive_keywords_dict'] = [var[1] for var in list(flood_filter_tuples)]
    input_df.loc[:,'negative_keywords_dict'] = [var[2] for var in list(flood_filter_tuples)]

    # Removing tenders from certain departments that are not related to flood management.
    idea_frm_tenders_df = input_df[(input_df.is_flood_tender=='True')&
                                    (~input_df.Department.isin(["Directorate of Agriculture and Assam Seed Corporation","Department of Handloom Textile and Sericulture"]))]

    print('Number of flood related tenders filtered: ', idea_frm_tenders_df.shape[0])
    if idea_frm_tenders_df.shape[0]==0:
        continue

    # Classify tenders based on Monsoons
    for index, row in idea_frm_tenders_df.iterrows():
        monsoon = "" 
        published_date = dateutil.parser.parse(row['Published Date'])
        if 1 <= published_date.month <= 5:
            monsoon = "Pre-Monsoon"
            if published_date.month == 5 and published_date.day > 14:
                monsoon = "Monsoon"
        elif 6 <= published_date.month <= 10:
            monsoon = "Monsoon"
            if published_date.month == 10 and published_date.day > 14:
                monsoon = "Post-Monsoon"
        else:
            monsoon = "Post-Monsoon"
        idea_frm_tenders_df.loc[index, "Season"] = monsoon

    # identify scheme related information
    schemes_identified = []
    scheme_kw = {'ridf','sdrf','sopd','cidf','ltif'}
    for idx, row in idea_frm_tenders_df.iterrows():
        tender_slug = row['tender_title']+' '+row['tender_externalreference']+' '+row['Work Description']
        tender_slug = re.sub(r'[^a-zA-Z0-9 \n\.]', ' ', tender_slug).lower()

        tender_slug = set(re.split(r'[-.,()_\s/]\s*',tender_slug))
        try:
            schemes_identified.append(list(tender_slug & scheme_kw)[0].upper())
        except:
            schemes_identified.append('')

    idea_frm_tenders_df.loc[:,'Scheme'] = schemes_identified

    # EROSION RELATED TENDERS
    EROSION_KEYWORDS = ['anti erosion', 'ae', 'a/e', 'a e', 'erosion', 'eroded', 'erroded', 'errosion']
    for index, row in idea_frm_tenders_df.iterrows():
        tender_slug = str(row['tender_externalreference']) + ' ' + str(row['tender_title']) + ' ' + str(row['Work Description'])
        tender_slug = re.sub(r'[^a-zA-Z0-9 \n\.]', ' ', tender_slug)

        is_present = [len(re.findall(r"\b%s\b" % kw.lower(), tender_slug.lower())) for kw in EROSION_KEYWORDS]
        if sum(is_present)>0:
            idea_frm_tenders_df.loc[index, "Erosion"] = True
        else:
            idea_frm_tenders_df.loc[index, "Erosion"] = False
    
    # ROADS, BRIDGES EMBANKMENTS RELATED TENDERS
    ROADS_BRIDGES_EMBANKMENTS_KEYWORDS = ['roads', 'bridges', 'road', 'bridge', 'storm water drain' ,'drain',
                                          'box cul', 'box culvert', 'box culv', 'culvert' ,'embankment', 'embkt',
                                          'river bank protection', 'bund', 'bunds', 'bundh', 'bank protection', 'dyke',
                                          'dyke wall', 'dyke walls', 'silt', 'siltation', 'sluice', 'breach']
    for index, row in idea_frm_tenders_df.iterrows():
        tender_slug = str(row['tender_externalreference']) + ' ' + str(row['tender_title']) + ' ' + str(row['Work Description'])
        tender_slug = re.sub(r'[^a-zA-Z0-9 \n\.]', ' ', tender_slug)

        is_present = [len(re.findall(r"\b%s\b" % kw.lower(), tender_slug.lower())) for kw in ROADS_BRIDGES_EMBANKMENTS_KEYWORDS]
        if sum(is_present)>0:
            idea_frm_tenders_df.loc[index, "Roads_Bridges_Embkt"] = True
        else:
            idea_frm_tenders_df.loc[index, "Roads_Bridges_Embkt"] = False

    

    idea_frm_tenders_df.to_csv(os.getcwd()+r'/flood-data-ecosystem-Himachal-Pradesh/Sources/TENDERS/data/flood_tenders/'+filename,
                            encoding='utf-8',
                            index=False)
                            
    
# Add explanation for this piece of code
data_path = os.getcwd() + r'/flood-data-ecosystem-Himachal-Pradesh/Sources/TENDERS/data/'
csvs = glob.glob(data_path+r'/flood_tenders/*.csv')
dfs=[]
for csv in csvs:
    csv = csv.replace("//", "/")
    csv = csv.replace("\\", "/")
    month = csv.split(r'/')[-1][:7]
    df = pd.read_csv(csv)
    df['month'] = month
    dfs.append(df)

idea_frm_tenders_df = pd.concat(dfs)
idea_frm_tenders_df.to_csv(data_path+'flood_tenders_all.csv', index=False)
